# Tutorial 3: Joining Tables (Lecture 9, Ch. 17)

Every query we wrote in the Lecture 8 tutorial touched exactly one table at a time. That was fine as far as it went, but it meant we could never answer a question like *which customer placed this order*, because the customer's name and the order live in two different tables.

This is where the foreign keys we set up in Tutorial 1 finally pay off. A join follows a foreign key from one table to another, so we can pull columns from both at once.

Run this notebook after the Lecture 8 tutorials, since it queries the tables those build.

In [ ]:
import pymysql

DB_HOST = "mysql"          # use "localhost" if running outside the jupyter container
DB_PORT = 3306
DB_USER = "root"
DB_PASSWORD = "OSUGoPokes"
DB_NAME = "iem4723_mysql_tutorial"

conn = pymysql.connect(
    host=DB_HOST,
    port=DB_PORT,
    user=DB_USER,
    password=DB_PASSWORD,
    database=DB_NAME,
)
cursor = conn.cursor()

In [ ]:
def run(label, query, params=None):
    """Run a query and print whatever comes back, under a label."""
    cursor.execute(query, params or ())
    rows = cursor.fetchall()
    print(f"--- {label} ---")
    if not rows:
        print("(no rows)")
    for row in rows:
        print(row)
    print()

## The Traditional Join

Remember the two syntaxes from lecture? We will start with the older one, because it is the one that shows most plainly what a join actually *is*.

You list both tables after `FROM`, and then the `WHERE` clause does the real work. The condition `c.Customer_Num = co.Customer_Num` is what pairs each order up with the customer who placed it. Leave that condition out and you get every customer paired with every order, which is almost never what you want.

Notice the aliases, *c* and *co*. Once two tables are in play, a bare `Customer_Num` is ambiguous, so we qualify it.

In [ ]:
traditional_join_query = """
SELECT c.Customer_FirstName, c.Customer_LastName, co.Order_Num, co.Order_Date
FROM customer c, customerorder co
WHERE c.Customer_Num = co.Customer_Num
ORDER BY co.Order_Num;
"""
run("Traditional join: who placed which order", traditional_join_query)

## The Same Join in SQL-92 Syntax

Now the newer form. The join condition moves out of `WHERE` and into an `ON` clause attached to the `JOIN` itself.

The result is identical. What changes is readability: the `WHERE` clause is now free to hold only the row filtering, instead of mixing the filtering together with the table plumbing.

In [ ]:
join_on_query = """
SELECT c.Customer_FirstName, c.Customer_LastName, co.Order_Num, co.Order_Date
FROM customer c JOIN customerorder co
  ON c.Customer_Num = co.Customer_Num
ORDER BY co.Order_Num;
"""
run("SQL-92 JOIN ... ON", join_on_query)

## NATURAL JOIN, and Why to Be Careful With It

A `NATURAL JOIN` asks the DBMS to find the columns that share a name across both tables and join on all of them, with no condition written at all.

Here it works, because *Customer_Num* is the only name the two tables have in common. That is also exactly why it is risky. If someone later adds a column to one table whose name happens to match a column in the other, this query silently starts joining on that too, and quietly returns different rows. Nothing errors out. For that reason most people reach for `ON` or `USING` in anything they intend to keep.

In [ ]:
natural_join_query = """
SELECT Customer_FirstName, Customer_LastName, Order_Num, Order_Date
FROM customer NATURAL JOIN customerorder
ORDER BY Order_Num;
"""
run("NATURAL JOIN", natural_join_query)

## JOIN ... USING

`USING` sits between the two. We still name the column to join on, so nothing happens by accident, but we only write it once instead of spelling out `c.Customer_Num = co.Customer_Num`.

It only works when the column has the *same name* in both tables. When the names differ, which happens often enough with foreign keys, you are back to `ON`.

In [ ]:
join_using_query = """
SELECT Customer_FirstName, Customer_LastName, Order_Num, Order_Date
FROM customer JOIN customerorder USING (Customer_Num)
ORDER BY Order_Num;
"""
run("JOIN ... USING", join_using_query)

## Joining Three Tables

A join works on two tables at a time, so what do we do when the answer needs three? We chain them. Each additional table brings its own join condition.

Here we want the item descriptions on a given order. *customerorder* knows the order, *item* knows the description, and *order_item* is the table in the middle that connects them. Without that middle table there is no path between the other two.

In [ ]:
three_table_query = """
SELECT co.Order_Num, i.Item_Description, oi.Item_Qty
FROM customerorder co, order_item oi, item i
WHERE co.Order_Num = oi.Order_Num
  AND oi.Item_ID = i.Item_ID
  AND co.Order_Num = %s;
"""
run("Three tables: what was on order 16642826", three_table_query, ("16642826",))

And the same thing in SQL-92 form, chaining one `JOIN ... ON` after another. Four tables now, since we have gone one step further back to pick up the customer's name as well.

In [ ]:
four_table_query = """
SELECT c.Customer_LastName, co.Order_Num, i.Item_Description, oi.Item_Qty
FROM customer c
  JOIN customerorder co ON c.Customer_Num = co.Customer_Num
  JOIN order_item oi    ON co.Order_Num = oi.Order_Num
  JOIN item i           ON oi.Item_ID = i.Item_ID
WHERE c.Customer_LastName = %s;
"""
run("Four tables: everything Felton ordered", four_table_query, ("Felton",))

## Joining on More Than One Column

Lecture covered joining on a concatenated key, where a single column is not enough to line the rows up.

Our *order_item* table has a concatenated primary key of *Order_Num* and *Item_ID* together, so here is a question that needs two columns in the join condition. Which shipments delivered exactly the quantity that someone had ordered of that same item? Matching on *Item_ID* alone would not answer it; we have to match the quantity as well.

In [ ]:
two_column_join_query = """
SELECT DISTINCT si.Shipment_Num, oi.Order_Num, i.Item_Description, oi.Item_Qty
FROM shipment_info si, order_item oi, item i
WHERE si.Item_ID = oi.Item_ID
  AND si.Item_Qty = oi.Item_Qty
  AND oi.Item_ID = i.Item_ID
ORDER BY si.Shipment_Num;
"""
run("Two column join: shipped quantity matched ordered quantity", two_column_join_query)

## Joining a Table to Itself

This one looks strange the first time. We bring *item* into the query **twice**, under two different aliases, and join it to itself.

Why would we ever do that? Because the question compares rows of one table against other rows of the same table. Here we want items that share a stock location with some other item. *i1* is the item we are reporting, *i2* is the one it shares a shelf with.

The `<>` at the end is doing essential work. Without it, every row matches itself, since an item trivially shares a location with itself, and we would get the entire table back.

In [ ]:
self_join_query = """
SELECT DISTINCT i1.Item_ID, i1.Item_Description, i1.Item_StockLocation
FROM item i1, item i2
WHERE i1.Item_StockLocation = i2.Item_StockLocation
  AND i1.Item_ID <> i2.Item_ID
ORDER BY i1.Item_StockLocation, i1.Item_ID;
"""
run("Self join: items sharing a stock location", self_join_query)

Here is the other shape of self join from lecture, finding the customers who ordered *two particular items*. We bring *order_item* in twice, once for each item, and require both to sit on the same order.

In [ ]:
two_items_query = """
SELECT DISTINCT c.Customer_Num, c.Customer_FirstName, c.Customer_LastName
FROM order_item t1, order_item t2, customerorder co, customer c
WHERE t1.Item_ID = %s
  AND t2.Item_ID = %s
  AND t1.Order_Num = t2.Order_Num
  AND t1.Order_Num = co.Order_Num
  AND co.Customer_Num = c.Customer_Num;
"""
run("Customers who ordered both 14 and 25 on one order", two_items_query, ("14", "25"))

### Closing Up

Same as always, close *cursor* first and then *conn*.

In [ ]:
cursor.close()
conn.close()